# Airtel Enterprise Customer Churn & Service Intelligence — Exploratory Data Analysis

**Dataset:** Synthetic Airtel Business-style enterprise customer dataset (25,000 records)
**Disclaimer:** This project uses a synthetically generated enterprise customer dataset inspired by Airtel Business service categories and is intended for analytical and educational purposes. It does not use or reference any real Airtel customer data.

## 1. Business Problem

Airtel Business (hypothetically, for this project) wants to understand:
- Which enterprise customers are at risk of churning, and why
- Which services, cities, states, and industries have the highest churn
- How much revenue is at risk from churn
- What the retention team should do about it

This notebook covers exploratory analysis. Prediction and retention scoring are covered in `02_Churn_Prediction_Model.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/airtel_enterprise_churn.csv')
print(f"Shape: {df.shape}")
df.head()

## 2. Dataset Overview

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data Dictionary

See `../reports/DATA_DICTIONARY.md` for the full column reference. Quick summary of column groups:

In [ ]:
groups = {
    'Customer Info': ['Customer_ID','Company_Name','Company_Size','Industry','Customer_Type','Customer_Segment','Years_With_Airtel'],
    'Geography': ['State','City','Region','Pincode_Zone'],
    'Service Usage': ['Number_of_Services','Primary_Service','Monthly_Bill','Annual_Contract_Value','Contract_Type'],
    'Service Quality': ['Network_Uptime','Downtime_Hours','SLA_Breaches','Support_Response_Hours'],
    'Experience': ['Customer_Satisfaction_Score','NPS_Score'],
    'Competitor': ['Competitor_Considered','Competitor_Threat_Level'],
    'Churn': ['Churn','Churn_Date','Churn_Reason','Churn_Category'],
}
for g, cols in groups.items():
    print(f"{g}: {cols}")

## 4. Data Quality Check

In [ ]:
print("Duplicate Customer_IDs:", df['Customer_ID'].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())

## 5. Missing Values

Note: `Churn_Date`, `Churn_Reason`, `Churn_Category` are expected to be null for active customers, and `Competitor_Offer` is expected to be null ("None") where the customer never considered a competitor. These are not data quality issues.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

## 6. Duplicate Check

In [ ]:
print("No duplicate customers." if df['Customer_ID'].duplicated().sum() == 0 else "DUPLICATES FOUND")

## 7. Descriptive Statistics — Key Numeric Fields

In [ ]:
key_numeric = ['Years_With_Airtel','Monthly_Bill','Annual_Contract_Value','Network_Uptime',
               'Downtime_Hours','Support_Response_Hours','Customer_Satisfaction_Score','NPS_Score']
df[key_numeric].describe().T

## 8. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df['Company_Size'].value_counts().plot(kind='bar', ax=axes[0,0], title='Customers by Company Size', color='#e60000')
df['Customer_Segment'].value_counts().plot(kind='bar', ax=axes[0,1], title='Customers by Segment', color='#e60000')
df['Industry'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], title='Top 10 Industries by Customer Count', color='#e60000')
df['Contract_Type'].value_counts().plot(kind='bar', ax=axes[1,1], title='Customers by Contract Type', color='#e60000')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Network_Uptime'], bins=40, ax=axes[0], color='#e60000')
axes[0].set_title('Distribution of Network Uptime (%)')
axes[0].set_xlabel('Network Uptime (%)')
sns.histplot(df['Customer_Satisfaction_Score'], bins=20, ax=axes[1], color='#e60000')
axes[1].set_title('Distribution of Customer Satisfaction Score')
axes[1].set_xlabel('Satisfaction (1-10)')
plt.tight_layout()
plt.show()

## 9. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Churn', y='Downtime_Hours', ax=axes[0])
axes[0].set_title('Downtime Hours: Active vs Churned')
axes[0].set_xticklabels(['Active', 'Churned'])
sns.boxplot(data=df, x='Churn', y='Customer_Satisfaction_Score', ax=axes[1])
axes[1].set_title('Satisfaction Score: Active vs Churned')
axes[1].set_xticklabels(['Active', 'Churned'])
plt.tight_layout()
plt.show()

## 10. Churn Analysis

### 10.1 Overall churn rate

In [ ]:
churn_rate = df['Churn'].mean() * 100
print(f"Total customers: {len(df):,}")
print(f"Churned: {df['Churn'].sum():,}")
print(f"Active: {(df['Churn']==0).sum():,}")
print(f"Overall churn rate: {churn_rate:.2f}%")

fig, ax = plt.subplots(figsize=(5,5))
df['Churn'].value_counts().rename({0:'Active',1:'Churned'}).plot(
    kind='pie', autopct='%1.1f%%', colors=['#2ca02c','#e60000'], ax=ax, ylabel='')
ax.set_title('Active vs Churned Customers')
plt.show()

### 10.2 Churn by reason

In [ ]:
reason_counts = df[df['Churn']==1]['Churn_Reason'].value_counts()
reason_pct = (reason_counts / reason_counts.sum() * 100).round(2)
reason_table = pd.DataFrame({'Customers': reason_counts, 'Churn %': reason_pct})
display(reason_table)

fig, ax = plt.subplots(figsize=(10,6))
reason_counts.plot(kind='barh', ax=ax, color='#e60000')
ax.set_title('Churned Customers by Reason')
ax.set_xlabel('Number of Customers')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 10.3 Churn by grouped category

In [ ]:
category_counts = df[df['Churn']==1]['Churn_Category'].value_counts()
fig, ax = plt.subplots(figsize=(6,6))
category_counts.plot(kind='pie', autopct='%1.1f%%', ax=ax, ylabel='')
ax.set_title('Churn Reasons Grouped by Category')
plt.show()
category_counts

### 10.4 Churn by customer segment and industry

In [ ]:
seg_churn = df.groupby('Customer_Segment')['Churn'].agg(['count','sum','mean'])
seg_churn.columns = ['Total','Churned','Churn_Rate']
seg_churn['Churn_Rate'] = (seg_churn['Churn_Rate']*100).round(2)
display(seg_churn.sort_values('Churn_Rate', ascending=False))

industry_churn = df.groupby('Industry')['Churn'].agg(['count','sum','mean'])
industry_churn.columns = ['Total','Churned','Churn_Rate']
industry_churn['Churn_Rate'] = (industry_churn['Churn_Rate']*100).round(2)
display(industry_churn.sort_values('Churn_Rate', ascending=False))

fig, ax = plt.subplots(figsize=(10,6))
industry_churn.sort_values('Churn_Rate', ascending=False)['Churn_Rate'].plot(kind='barh', ax=ax, color='#e60000')
ax.set_title('Churn Rate by Industry')
ax.set_xlabel('Churn Rate (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Geographic Analysis

### 11.1 State-level churn

In [ ]:
state_stats = df.groupby('State').agg(
    Total=('Customer_ID','count'), Churned=('Churn','sum'))
state_stats['Active'] = state_stats['Total'] - state_stats['Churned']
state_stats['Churn_Rate'] = (state_stats['Churned']/state_stats['Total']*100).round(2)
state_stats['Revenue_at_Risk'] = df[df.Churn==1].groupby('State')['Annual_Contract_Value'].sum()
state_stats['Revenue_at_Risk'] = state_stats['Revenue_at_Risk'].fillna(0)

print("Top 10 states by CHURN RATE:")
display(state_stats.sort_values('Churn_Rate', ascending=False).head(10))

print("\nTop 10 states by NUMBER of churned customers:")
display(state_stats.sort_values('Churned', ascending=False).head(10))

**Why these two rankings differ:** a state can have a high churn *rate* with relatively few total customers (a handful of small states show this), while a large state like Maharashtra or Karnataka can have a comparatively average churn rate but the highest *absolute* number of churned customers simply because it carries the largest customer base. For revenue-at-risk prioritization, absolute churned count and revenue-at-risk matter more than rate alone; for identifying systemic service problems, rate is the better signal.

### 11.2 City-level churn

In [ ]:
city_stats = df.groupby('City').agg(Total=('Customer_ID','count'), Churned=('Churn','sum'))
city_stats['Churn_Rate'] = (city_stats['Churned']/city_stats['Total']*100).round(2)

print("Top 15 cities by churn RATE (min 50 customers):")
display(city_stats[city_stats.Total>=50].sort_values('Churn_Rate', ascending=False).head(15))

print("\nTop 15 cities by churned VOLUME:")
display(city_stats.sort_values('Churned', ascending=False).head(15))

fig, ax = plt.subplots(figsize=(10,7))
city_stats.sort_values('Churned', ascending=False).head(15)['Churned'].plot(kind='barh', ax=ax, color='#e60000')
ax.set_title('Top 15 Cities by Number of Churned Customers')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Service Analysis

In [ ]:
services = ["Internet_Leased_Line","Dedicated_Internet","MPLS_VPN","SD_WAN","Broadband",
            "International_Private_Line","Airtel_Cloud","Multi_Cloud_Connect","Cloud_Backup",
            "Disaster_Recovery","Managed_Firewall","DDoS_Protection","Network_Security",
            "Secure_Internet","Zero_Trust","Business_Voice","CPaaS","Enterprise_Messaging",
            "Collaboration_Services","IoT_Services"]

rows = []
for s in services:
    sub = df[df[s]==1]
    rows.append({'Service': s, 'Customers': len(sub), 'Churn_Rate': round(sub['Churn'].mean()*100,2),
                 'Revenue_at_Risk': sub.loc[sub.Churn==1,'Annual_Contract_Value'].sum()})
service_churn = pd.DataFrame(rows).sort_values('Churn_Rate', ascending=False)
display(service_churn)

fig, ax = plt.subplots(figsize=(10,8))
service_churn.set_index('Service')['Churn_Rate'].plot(kind='barh', ax=ax, color='#e60000')
ax.set_title('Churn Rate by Airtel Service')
ax.set_xlabel('Churn Rate (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 13. Correlation Analysis

In [ ]:
drivers = ["Downtime_Hours","Number_of_Outages","SLA_Breaches","Support_Response_Hours",
           "Support_Resolution_Hours","Number_of_Complaints","Billing_Issues","Packet_Loss_Percentage",
           "Average_Latency_ms","Customer_Satisfaction_Score","NPS_Score","Network_Uptime",
           "Years_With_Airtel","Number_of_Services","Service_Quality_Score","Churn"]

corr_matrix = df[drivers].corr()
fig, ax = plt.subplots(figsize=(11,9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation Matrix — Churn Drivers')
plt.tight_layout()
plt.show()

print("Correlation with Churn (sorted by strength):")
print(corr_matrix['Churn'].drop('Churn').sort_values(key=abs, ascending=False))

**Reading the correlations:** Downtime hours, complaint counts, and SLA breaches show the strongest positive correlation with churn, while network uptime and customer satisfaction show the strongest negative correlation. This lines up with the churn reason breakdown above, where "Frequent Downtime" and "Slow Customer Support" are the two largest single reasons customers leave — the network/support quality signals aren't just correlated with churn, they're driving the majority of churn reasons customers actually cite.

## 14. Business Insights Summary

In [ ]:
print("="*70)
print("KEY BUSINESS INSIGHTS")
print("="*70)

worst_service = service_churn.iloc[0]
worst_state = state_stats.sort_values('Churn_Rate', ascending=False).iloc[0]
worst_industry = industry_churn.sort_values('Churn_Rate', ascending=False).iloc[0]
top_reason = reason_counts.index[0]
total_risk = df.loc[df.Churn==1, 'Annual_Contract_Value'].sum()

print(f"1. Overall churn rate: {churn_rate:.2f}% across {len(df):,} enterprise customers")
print(f"2. Biggest churn driver (by volume): {top_reason} ({reason_counts.iloc[0]} customers, {reason_pct.iloc[0]}%)")
print(f"3. Worst-performing service: {worst_service['Service']} ({worst_service['Churn_Rate']}% churn rate)")
print(f"4. Worst-performing state (by rate): {worst_state.name} ({worst_state['Churn_Rate']}% churn rate)")
print(f"5. Highest-risk industry: {worst_industry.name} ({worst_industry['Churn_Rate']}% churn rate)")
print(f"6. Total annual contract value at risk from churned customers: Rs {total_risk:,.0f}")
print(f"7. Avg downtime — churned: {df.loc[df.Churn==1,'Downtime_Hours'].mean():.1f}h vs active: {df.loc[df.Churn==0,'Downtime_Hours'].mean():.1f}h")
print(f"8. Avg satisfaction — churned: {df.loc[df.Churn==1,'Customer_Satisfaction_Score'].mean():.2f}/10 vs active: {df.loc[df.Churn==0,'Customer_Satisfaction_Score'].mean():.2f}/10")
print("="*70)
print("\nSee ../reports/business_insights.md for the full narrative writeup.")